Convert wo CV metadata

- pixel spacing
  - xy: from tif file
  - z: from MeasurementProtocol.xml
    -     <icm:ZSetting icm:Range="76.5000" icm:Slices="10" icm:BottomAbs_um="-25.8000" />

- channels: manually
- shapes: from tif file
- FOV center positions: from measurementprotocol.xml
      <icm:Well icm:UniqueID="203" icm:Number="4" icm:Column="4" icm:Row="1" icm:IsIgnore="false" icm:IsEnabled="true" icm:IsAnalysis="false">
        <icm:CenterCoord icm:X="-22.48" icm:Y="32.06" />
        <icm:Areas />
      </icm:Well>
- image data: from tiff files


In [6]:
from pathlib import Path
import os
import tifffile

# input_dir = Path("/Volumes/scuanalysis-1/LAF/Sant/P2001_Hierlemann_00013516_wo_CV_format/20251027T132950_10X")
input_dir = Path("/Volumes/scuanalysis/Hierlemann/carolina/Yokogawa_CQ3000/20250729T171334_20X_dry_ST2036Q10W")

In [7]:
# list wells (starts at 1)
wells = sorted(list((input_dir / "Image").glob("Well*")))
first_tiff_file_path = wells[0] / os.listdir(wells[0])[0]
first_tiff_file_path
well_indices = [int(well.name.replace("Well", "")) - 1 for well in wells]
well_indices

[25,
 26,
 27,
 28,
 31,
 32,
 33,
 34,
 37,
 38,
 39,
 40,
 43,
 44,
 45,
 46,
 49,
 50,
 51,
 52,
 55,
 56,
 57,
 58,
 61,
 62,
 63,
 64,
 67,
 68,
 69,
 70,
 73,
 74,
 75,
 76,
 79,
 80,
 81,
 82,
 85,
 86,
 87,
 88,
 91,
 92,
 93,
 94]

In [8]:
import xmltodict

mprot_dict = xmltodict.parse(open(input_dir / "MeasurementProtocol.xml", 'r').read())
#mprot_dict['icm:MeasurementProtocol']['icm:ImagingProtocol']['icm:Wells']

In [9]:
# <icm:ImagingResult icm:StructVersion="2.0" xmlns:icm="http://www.yokogawa.co.jp/LSC/ICMSchema/1.0">
#   <icm:FileInformation icm:Name="ImagingResult.xml" icm:Version="2.0" icm:CreateTime="2025-07-29T17:13:34.9542688+02:00" icm:CreateUser="cquser" icm:LastUpdateTime="2025-07-30T02:01:17.5882598+02:00" icm:LastUpdateUser="cquser"></icm:FileInformation>
#   <icm:ResultInfo icm:BeginTime="2025-07-29T17:13:34.2128961+02:00" icm:EndTime="2025-07-30T02:01:17.5080869+02:00" icm:Result="Completed" icm:Software="CQ Software" icm:SoftwareVersion="R1.02.01.02" icm:PlateID="NoPlateID">
#     <icm:ProtocolPath>MeasurementProtocol.xml</icm:ProtocolPath>
#   </icm:ResultInfo>
#   <icm:MachineInfo icm:ProductID="-" icm:SerialNumber="9021B9199" icm:FirmwareVersion="1.01.01.04" icm:AutoFocusRevision="4" />
#   <icm:ImageDataInfo icm:FolderDividingMethod="ByWell" icm:IsFlatFieldCalibration="false" icm:IsGeometricCalibration="false" icm:LastTimePoint="1" icm:TotalTimePoint="1">
#     <icm:DimensionsInfo>
#       <icm:W icm:Min="1" icm:Max="48" />
#       <icm:A icm:Min="1" icm:Max="1" />
#       <icm:F icm:Min="1" icm:Max="100" />
#       <icm:T icm:Min="1" icm:Max="1" />
#       <icm:Z icm:Min="1" icm:Max="5" />
#       <icm:C icm:Min="1" icm:Max="4" />
#     </icm:DimensionsInfo>

imaging_result_dict = xmltodict.parse(open(input_dir / "ImagingResult.xml", 'r').read())
dims_info = imaging_result_dict['icm:ImagingResult']['icm:ImageDataInfo']['icm:DimensionsInfo']
dims_shape = {k[-1]: int(v['@icm:Max']) - int(v['@icm:Min']) + 1 for k, v in dims_info.items()}
dims_shape

{'W': 48, 'A': 1, 'F': 100, 'T': 1, 'Z': 5, 'C': 4}

In [10]:
import tifffile

def read_tiff_voxel_size_and_shape(file_path):
    """
    Implemented based on information found in https://pypi.org/project/tifffile

    Taken from https://forum.image.sc/t/reading-pixel-size-from-image-file-with-python/74798/4
    """

    def _xy_voxel_size(tags, key):
        assert key in ['XResolution', 'YResolution']
        if key in tags:
            num_pixels, units = tags[key].value
            return units / num_pixels
        # return default
        return 1.

    with tifffile.TiffFile(file_path) as tiff:
        image_metadata = tiff.imagej_metadata
        if image_metadata is not None:
            z = image_metadata.get('spacing', 1.)
        else:
            # default voxel size
            z = 1.

        tags = tiff.pages[0].tags
        # parse X, Y resolution
        y = _xy_voxel_size(tags, 'YResolution')
        x = _xy_voxel_size(tags, 'XResolution')
        # return voxel size

        shape = tiff.pages[0].shape
        return [z, y, x], shape

spacing_zyx, shape = read_tiff_voxel_size_and_shape(first_tiff_file_path)

z_settings = mprot_dict['icm:MeasurementProtocol']['icm:ImagingProtocol']['icm:ZSetting']
z_settings

spacing_z = float(z_settings['@icm:Range']) / (int(z_settings['@icm:Slices']) - 1)
spacing_z

# spacing = {'z': spacing_z, 'y': spacing_zyx[1] * 2.54, 'x': spacing_zyx[2] * 2.54}
spacing = {'z': spacing_z, 'y': spacing_zyx[1], 'x': spacing_zyx[2]}

shape = {'y': shape[0], 'x': shape[1]}

spacing, shape

({'z': 6.1375, 'y': 0.010416666666666666, 'x': 0.010416666666666666},
 {'y': 1400, 'x': 1400})

In [11]:
import xmltodict

mprot_dict = xmltodict.parse(open(input_dir / "MeasurementProtocol.xml", 'r').read())
fields = mprot_dict['icm:MeasurementProtocol']['icm:ImagingProtocol']['icm:Wells']['icm:SameSettingWell']['icm:Areas']['icm:Area']['icm:Fields']['icm:Field']
if type(fields) is not list:
    fields = [fields]

field_poss = [{'x': float(f['@icm:X']), 'y': float(f['@icm:Y'])} for f in fields]
field_poss

[{'x': -2015.2, 'y': 2081.6},
 {'x': -1562.8, 'y': 2081.6},
 {'x': -1110.4, 'y': 2081.6},
 {'x': -658.0, 'y': 2081.6},
 {'x': -205.6, 'y': 2081.6},
 {'x': 247.0, 'y': 2081.6},
 {'x': 699.4, 'y': 2081.6},
 {'x': 1151.8, 'y': 2081.6},
 {'x': 1604.2, 'y': 2081.6},
 {'x': 2056.6, 'y': 2081.6},
 {'x': -2015.2, 'y': 1629.2},
 {'x': -1562.8, 'y': 1629.2},
 {'x': -1110.4, 'y': 1629.2},
 {'x': -658.0, 'y': 1629.2},
 {'x': -205.6, 'y': 1629.2},
 {'x': 247.0, 'y': 1629.2},
 {'x': 699.4, 'y': 1629.2},
 {'x': 1151.8, 'y': 1629.2},
 {'x': 1604.2, 'y': 1629.2},
 {'x': 2056.6, 'y': 1629.2},
 {'x': -2015.2, 'y': 1176.8},
 {'x': -1562.8, 'y': 1176.8},
 {'x': -1110.4, 'y': 1176.8},
 {'x': -658.0, 'y': 1176.8},
 {'x': -205.6, 'y': 1176.8},
 {'x': 247.0, 'y': 1176.8},
 {'x': 699.4, 'y': 1176.8},
 {'x': 1151.8, 'y': 1176.8},
 {'x': 1604.2, 'y': 1176.8},
 {'x': 2056.6, 'y': 1176.8},
 {'x': -2015.2, 'y': 724.4},
 {'x': -1562.8, 'y': 724.4},
 {'x': -1110.4, 'y': 724.4},
 {'x': -658.0, 'y': 724.4},
 {'x': -205.

In [12]:
well_index_to_col_and_row = {}

for w in mprot_dict['icm:MeasurementProtocol']['icm:ImagingProtocol']['icm:Wells']['icm:Well']:
    well_index = int(w['@icm:Number']) - 1
    col = int(w['@icm:Column'])
    row_index = int(w['@icm:Row'])
    row = chr(ord('A') + row_index - 1)
    well_index_to_col_and_row[well_index] = {'col': col, 'row': row}
    
well_index_to_col_and_row

{0: {'col': 1, 'row': 'A'},
 1: {'col': 2, 'row': 'A'},
 2: {'col': 3, 'row': 'A'},
 3: {'col': 4, 'row': 'A'},
 4: {'col': 5, 'row': 'A'},
 5: {'col': 6, 'row': 'A'},
 6: {'col': 7, 'row': 'A'},
 7: {'col': 8, 'row': 'A'},
 8: {'col': 9, 'row': 'A'},
 9: {'col': 10, 'row': 'A'},
 10: {'col': 11, 'row': 'A'},
 11: {'col': 12, 'row': 'A'},
 12: {'col': 1, 'row': 'B'},
 13: {'col': 2, 'row': 'B'},
 14: {'col': 3, 'row': 'B'},
 15: {'col': 4, 'row': 'B'},
 16: {'col': 5, 'row': 'B'},
 17: {'col': 6, 'row': 'B'},
 18: {'col': 7, 'row': 'B'},
 19: {'col': 8, 'row': 'B'},
 20: {'col': 9, 'row': 'B'},
 21: {'col': 10, 'row': 'B'},
 22: {'col': 11, 'row': 'B'},
 23: {'col': 12, 'row': 'B'},
 24: {'col': 1, 'row': 'C'},
 25: {'col': 2, 'row': 'C'},
 26: {'col': 3, 'row': 'C'},
 27: {'col': 4, 'row': 'C'},
 28: {'col': 5, 'row': 'C'},
 29: {'col': 6, 'row': 'C'},
 30: {'col': 7, 'row': 'C'},
 31: {'col': 8, 'row': 'C'},
 32: {'col': 9, 'row': 'C'},
 33: {'col': 10, 'row': 'C'},
 34: {'col': 11, 

In [13]:
# extract well, field, time, z and channel info from filenames:
# e.g. W0028F0026T0001Z005C4.tif

def extract_coord(filename):
    base = os.path.basename(filename)
    well = int(base[1:5])
    field = int(base[6:10])
    time = int(base[11:15])
    z = int(base[16:19])
    channel = int(base[20:21])
    return {
        "well": well,
        "field": field,
        "time": time,
        "z": z,
        "channel": channel
    }

# build filename from well, field, time, z and channel:
# e.g. W0028F0026T0001Z005C4.tif

def get_tif_file_name(well, field, time, z, channel):
    return f"W{well:04d}F{field:04d}T{time:04d}Z{z:03d}C{channel}.tif"

get_tif_file_name(well=28, field=26, time=1, z=5, channel=4)

extract_coord("W0028F0026T0001Z005C4.tif")

{'well': 28, 'field': 26, 'time': 1, 'z': 5, 'channel': 4}

## Create OME-Zarr

In [14]:
from ngio import ImageInWellPath, create_empty_plate

# output_zarr_path = Path("./data/test_plate.zarr")
# output_zarr_path = Path("/Volumes/scuanalysis-1/LAF/Sant/P2001_Hierlemann_00013516_wo_CV_format/20251027T132950_10X_2.ome.zarr")
# output_zarr_path = Path("/Volumes/bsse_shared_temp/SCF/Marvin/ome_zarr_tests/20251027T132950_10X.ome.zarr")
output_zarr_path = Path("/Volumes/bsse_shared_temp/SCF/Marvin/ome_zarr_tests/carolina1.ome.zarr")

test_plate = create_empty_plate(
    store=output_zarr_path,
    name="Test Plate",
    images=[
        # ImageInWellPath(row="A", column="01", path="0"),
        # ImageInWellPath(row="A", column="02", path="0"),
        # ImageInWellPath(row="A", column="02", path="1", acquisition_id=1),
        ImageInWellPath(
            row=well_index_to_col_and_row[well_index]['row'],
            column=f"{well_index_to_col_and_row[well_index]['col']:02d}",
            path=str(field_index))
        for well_index in well_indices[:1]
        for field_index in range(len(field_poss[:10]))
    ],
    overwrite=True,
    # version=
)

print(test_plate)
print(f"Rows: {test_plate.rows}, Columns: {test_plate.columns}")

Plate([rows x columns] (1 x 1))
Rows: ['C'], Columns: ['02']


In [22]:
def process_batch_using_joblib(func, block_ids, n_jobs=4):
    """
    A batch function that uses joblib for parallel processing.
    1. func: function to apply to each block_id
    2. block_ids: list of block IDs to process
    3. n_jobs: number of parallel jobs to run
    """

    try:
        from joblib import Parallel, delayed
    except ImportError:
        raise ImportError("Please install joblib to use this function.")

    Parallel(n_jobs=n_jobs)(
        delayed(func)(block_id) for block_id in block_ids
    )
    return


In [25]:
from multiview_stitcher import ngff_utils, misc_utils
from multiview_stitcher import spatial_image_utils as si_utils
import dask.array as da
from dask import delayed

for well_index in well_indices[:1]:
    for field_index in range(len(field_poss[:2])):

        # load all z slices and channels as a dask array
        channels = []
        for ch in range(dims_shape['C']):
            stack = []
            for z in range(dims_shape['Z']):
                stack.append(
                    delayed(lambda x: tifffile.imread(x).T)(
                    # delayed(lambda x: tifffile.imread(x))(
                        input_dir / "Image" / f"Well{well_index + 1:04d}" / get_tif_file_name(
                            well=well_index + 1,
                            field=field_index + 1,
                            time=1,
                            z=z + 1,
                            channel=ch + 1,
                        ),
                        # key = z * dims_shape['C'] + ch
                    )
                )
            stack = da.stack([da.from_delayed(c, shape=(shape['y'], shape['x']), dtype='uint16') for c in stack], axis=0)
            channels.append(stack)
        channels = da.stack(channels, axis=0)  # shape: (C, Z, Y, X)

        sim = si_utils.get_sim_from_array(
            channels,
            dims=('c', 'z', 'y', 'x'),
            scale=spacing,
            translation={
                'x': field_poss[field_index]['x'],
                'y': field_poss[field_index]['y'],
                'z': 0,
            },
        )

        ngff_utils.write_sim_to_ome_zarr(
            sim,
            output_zarr_url=output_zarr_path / well_index_to_col_and_row[well_index]['row'] / f"{well_index_to_col_and_row[well_index]['col']:02d}" / str(field_index),
            ngff_version="0.4",
            n_batch=4,
            batch_func=process_batch_using_joblib
        )

Writing resolution level 0...


  0%|                                                                                                                                         | 0/5 [00:00<?, ?it/s]/Users/albertm/Library/Caches/rattler/cache/envs/CQ3000_converter-718051524441998400/envs/default/lib/python3.13/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'config'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/albertm/Library/Caches/rattler/cache/envs/CQ3000_converter-718051524441998400/envs/default/lib/python3.13/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'config'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/albertm/Library/Caches/rattler/cache/envs/CQ3000_converter-718051524441998400/envs/default/lib/python3.13/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'config'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/albertm/Library/

Writing resolution level 1...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  2.93it/s]


Writing resolution level 2...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  2.90it/s]


Writing resolution level 3...


  0%|                                                                                                                                         | 0/5 [00:00<?, ?it/s]


OSError: [Errno 16] Resource busy: '/Volumes/bsse_shared_temp/SCF/Marvin/ome_zarr_tests/carolina1.ome.zarr/C/02/0/3/.zarray.43918985a8e946f7bb16217e9d907e51.partial' -> '/Volumes/bsse_shared_temp/SCF/Marvin/ome_zarr_tests/carolina1.ome.zarr/C/02/0/3/.zarray'

In [150]:
import zarr
zarr.__version__

'2.18.7'

In [ ]:
import numpy as np
from ngio import create_ome_zarr_from_array

for well_index in well_indices:
    for field_index in range(len(field_poss)):
        # Create a random 3D array
        x = np.random.randint(0, 255, (16, shape['y'], shape['x']), dtype=np.uint8)

        # Save as OME-Zarr
        new_ome_zarr_image = create_ome_zarr_from_array(
            store=f"./data/test_plate.zarr/{well_index_to_col_and_row[well_index]['row']}{well_index_to_col_and_row[well_index]['col']:02d}/{field_index}",
            array=x,
            xy_pixelsize=spacing['x'],
            z_spacing=spacing['z']
        )   

# # Create a random 3D array
# x = np.random.randint(0, 255, (16, 128, 128), dtype=np.uint8)

# # Save as OME-Zarr
# new_ome_zarr_image = create_ome_zarr_from_array(
#     store="random_ome.zarr", 
#     array=x, 
#     xy_pixelsize=0.65, 
#     z_spacing=1.0
# )